# Thiết Kế Chuỗi Dịch Vụ Logistics Tự Động Có AI Ra Quyết Định
## Nghiệp vụ Kho (Inventory) — Dự đoán Nguy cơ Trễ Bàn Giao Vận Tải (Late Handover)
**Bộ môn E-Logistics & Supply Chain Analytics (CK10)**  
**Bộ dữ liệu:** Brazilian E-Commerce by Olist (99,441 đơn hàng, 2016–2018)  
**Thuật toán:** Decision Tree phân tầng tự hiện thực bằng **NumPy thuần** (Không phụ thuộc `scikit-learn` trong mã nguồn triển khai)  
**Tác giả:** Nhóm sinh viên thực hiện đề tài

---
### Tóm tắt cốt lõi (Executive Summary)
1. **Bài toán thực tế:** Trong chuỗi cung ứng thương mại điện tử nhiều người bán (marketplace), việc bàn giao hàng hóa cho đối tác vận chuyển muộn hơn thời hạn cam kết (`shipping_limit_date`) là nguyên nhân chính dẫn đến vỡ cam kết với khách hàng cuối (SLA).
2. **Định nghĩa nhãn chuẩn:**
   - $\text{late\_handover} = \mathbf{1}(\text{order\_delivered\_carrier\_date} > \text{shipping\_limit\_date})$
   - Khác biệt hoàn toàn với trễ giao khách: $\text{late\_delivery} = \mathbf{1}(\text{order\_delivered\_customer\_date} > \text{order\_estimated\_delivery\_date})$.
3. **Phát hiện quan trọng:**
   - Tín hiệu dự đoán nguy cơ trễ bàn giao hầu như toàn bộ đến từ **hành vi lịch sử của người bán** (`seller_han_rate`), chiếm hơn 65% độ quan trọng của cây. Các đặc trưng đơn hàng đơn lẻ (giá, khối lượng, giờ đặt) có năng lực dự đoán gần như ngẫu nhiên (AUC ~0.518).
   - Mô hình Cây quyết định (AUC ~0.655 - 0.667) không vượt trội hơn một luật phân loại đơn biến dựa trên lịch sử người bán (AUC ~0.671). Giá trị thực sự của AI nằm ở **năng lực phân tầng rủi ro (Triage)** và tích hợp đa điều kiện, cho phép Trưởng ca tập trung xử lý can thiệp vào top 15% đơn hàng nguy cơ cao nhất.
   - Trong mô phỏng hàng đợi kho: Quy tắc sắp xếp theo hạn bàn giao (EDD) giảm tỷ lệ trễ từ 7.2% xuống 0.8%. Mô hình kết hợp AI-EDD đạt 0.9%, không vượt hơn EDD vì AI dự báo trễ do người bán, trong khi hàng đợi tạo trễ do năng lực thông lượng kho. Ngược lại, nếu lạm dụng điểm AI để xếp thứ tự làm việc tuyệt đối ("AI level-first") mà bỏ qua deadline, tỷ lệ trễ sẽ tăng vọt lên 11.6% (tệ hơn cả FIFO truyền thống).

## 1. Thiết lập môi trường và Nạp thư viện (Bước 1)
Toàn bộ thuật toán cây quyết định, ma trận phân loại, chuẩn hóa, và mô phỏng được lập trình bằng thư viện số học **NumPy thuần**, loại trừ hoàn toàn việc dùng `scikit-learn` trong pipeline huấn luyện để đảm bảo tính minh bạch thuật toán.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thêm đường dẫn project root vào sys.path
BASE_DIR = os.path.abspath(os.path.join(os.getcwd()))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from src.data_prep import load_raw, build_order_table, add_features, select_cohort, time_split, fill_train_median, audit_raw, FEATURES
from src.model import DecisionTree, RandomForest, LogisticRegression, StandardScaler, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from src.weights import load_tree, export_tree_dict
from src.simulate import simulate, kpis, policies

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("Môi trường sẵn sàng. Nguyên tắc R1-R7 được bảo đảm.")

NumPy version: 1.26.4
Pandas version: 2.2.2
Môi trường sẵn sàng. Nguyên tắc R1-R7 được bảo đảm.


## 2. Khảo sát & Kiểm toán Dữ liệu Gốc (Bước 2 – 4)
Đọc 5 tệp CSV nguyên vẹn từ Olist, kiểm tra tính toàn vẹn mã băm SHA-256 (Rule R1: Dữ liệu raw không bao giờ bị ghi đè).

In [ ]:
raw = load_raw('data/raw')
print(f"Bảng Orders:    {raw['orders'].shape[0]:,} dòng")
print(f"Bảng Items:     {raw['items'].shape[0]:,} dòng")
print(f"Bảng Products:  {raw['products'].shape[0]:,} dòng")
print(f"Bảng Sellers:   {raw['sellers'].shape[0]:,} dòng")
print(f"Bảng Customers: {raw['customers'].shape[0]:,} dòng")

# Tải báo cáo kiểm toán đã xuất tại outputs/01_data_audit.json
with open('outputs/01_data_audit.json', 'r', encoding='utf-8') as f:
    audit_info = json.load(f)

print("\nPhễu lọc dữ liệu (Funnel):")
for step, cnt in audit_info['funnel'].items():
    if step != 'splits':
        print(f"  - {step}: {cnt:,} đơn")
    else:
        for sp_name, sp_cnt in cnt.items():
            print(f"     * {sp_name}: {sp_cnt:,} đơn")

Bảng Orders:    99,441 dòng
Bảng Items:     112,650 dòng
Bảng Products:  32,951 dòng
Bảng Sellers:   3,095 dòng
Bảng Customers: 99,441 dòng

Phễu lọc dữ liệu (Funnel):
  - 0_raw_orders: 99,441 đơn
  - 1_delivered_with_timestamps_1seller: 95,180 đơn
  - 2_cohort_20170101_to_20180801: 88,526 đơn
     * train_fit: 55,808 đơn
     * validation: 13,953 đơn
     * test: 18,765 đơn


## 3. Kỹ thuật Đặc trưng & Kiểm soát Rò rỉ Dữ liệu (Bước 5 – 10)
- **Knowledge-Time History:** Tỷ lệ trễ lịch sử của người bán `seller_han_rate` và tuyến đường `lane_late_rate` chỉ được tính toán từ các đơn hàng đã được giao hoàn tất **trước thời điểm duyệt đơn** (`order_approved_at`) của chính đơn đó.
- **Split theo chuỗi thời gian:** Cắt ngưỡng kiểm thử tại `2018-05-01`. Toàn bộ dữ liệu kiểm thử (Test) tuyệt đối không giao nhau về chỉ mục hay thời gian với tập Train/Validation.

In [ ]:
d_order = build_order_table(raw)
d_feat = add_features(d_order)
d_cohort = select_cohort(d_feat)

tr_fit, val_idx, test_idx = time_split(d_cohort)
print(f"Tập Huấn luyện (Train Fit): {len(tr_fit):,} đơn (từ 2017-01 đến 2018-02)")
print(f"Tập Thẩm định (Validation): {len(val_idx):,} đơn (từ 2018-02 đến 2018-04)")
print(f"Tập Kiểm thử (Test khóa):   {len(test_idx):,} đơn (từ 2018-05 đến 2018-07)")

# Điền giá trị khuyết thiếu bằng trung vị của tập Train
d_filled = fill_train_median(d_cohort, tr_fit)
print("Kiểm tra giá trị thiếu sau khi điền:", d_filled[FEATURES].isna().sum().sum())

Tập Huấn luyện (Train Fit): 55,808 đơn (từ 2017-01 đến 2018-02)
Tập Thẩm định (Validation): 13,953 đơn (từ 2018-02 đến 2018-04)
Tập Kiểm thử (Test khóa):   18,765 đơn (từ 2018-05 đến 2018-07)
Kiểm tra giá trị thiếu sau khi điền: 0


## 4. Mô hình Cơ sở & Nguyên lý "No Free Lunch" (Bước 11 – 13)
So sánh 7 mô hình và baseline trên cùng một tập phân chia thời gian:
1. **Dummy Baseline:** Dự đoán tỷ lệ tiên nghiệm $P(Y=1) = 7.7\%$.
2. **FIFO Baseline:** Điểm rủi ro tỷ lệ thuận với thời gian đến (đơn đến trước điểm cao).
3. **Slack Baseline:** Điểm rủi ro nghịch đảo với thời gian đệm tới hạn bàn giao.
4. **Seller History Rule:** Dự đoán duy nhất bằng đặc trưng `seller_han_rate`.
5. **Decision Tree (Order Only):** Cây chỉ dùng thông tin đơn hàng (bỏ qua thông tin người bán).
6. **Decision Tree (Core Features):** Cây quyết định kết hợp đầy đủ đặc trưng.
7. **Logistic Regression (NumPy GD + L2):** Hồi quy Logistic chuẩn hóa trọng số.
8. **Random Forest (NumPy Ensemble):** Rừng ngẫu nhiên 30 cây.

In [ ]:
with open('outputs/03_model_comparison.json', 'r', encoding='utf-8') as f:
    comp_data = json.load(f)

rows = []
for k, v in comp_data.items():
    name = k.split('_', 1)[1] if '_' in k else k
    rows.append({
        'STT': k[:1],
        'Mô hình': name,
        'Vai trò': v.get('role', ''),
        'Val ROC-AUC': v['val']['roc_auc'],
        'Test ROC-AUC': v['test']['roc_auc'],
        'Test PR-AUC': v['test']['pr_auc'],
        'Test Brier': v['test']['brier']
    })

comp_df = pd.DataFrame(rows)
print(comp_df.to_string(index=False))

STT             Mô hình                                               Vai trò  Val ROC-AUC  Test ROC-AUC  Test PR-AUC  Test Brier
  1               Dummy               Baseline sàn (đoán hằng số tỷ lệ train)       0.5000        0.5000       0.0747      0.0697
  2           Rule_FIFO              As-Is đề cương (đơn đến trước làm trước)       0.5534        0.4646       0.0671      0.3656
  3          Rule_Slack        Baseline kinh điển (slack ngắn nhất làm trước)       0.4813        0.4866       0.0722  23816.3746
  4 Rule_Seller_History  Baseline mạnh 1 đặc trưng (tỷ lệ trễ lịch sử seller)       0.7728        0.6712       0.1514      0.0680
  5     Tree_Order_Only            Cây không lịch sử (chỉ đặc trưng đơn hàng)       0.5348        0.5183       0.0795      0.0743
  6 Logistic_Regression Baseline tuyến tính (chuẩn hóa Z-score, class_weight)       0.7463        0.6833       0.1580      0.2530
  7  Decision_Tree_Core  Mô hình lõi theo đề (depth 5, leaf 200, thuần NumPy)       0.7538

## 5. Phân tích Bias – Variance qua Độ Sâu Cây (Bước 13)
Đánh giá hiện tượng quá khớp (Overfitting) khi tăng độ sâu `max_depth` từ 1 đến 12.

In [ ]:
with open('outputs/03_bias_variance.json', 'r', encoding='utf-8') as f:
    bv_data = json.load(f)

plt.figure(figsize=(9, 4.5))
plt.plot(bv_data['depths'], bv_data['train_roc_auc'], marker='o', label='Train Fit AUC')
plt.plot(bv_data['depths'], bv_data['val_roc_auc'], marker='s', label='Validation AUC')
plt.plot(bv_data['depths'], bv_data['test_roc_auc'], marker='^', label='Test AUC')
plt.axvline(x=5, color='r', linestyle='--', label='Độ sâu chọn lọc (depth=5-6)')
plt.xlabel('Độ sâu cây (max_depth)', fontsize=11)
plt.ylabel('ROC-AUC Score', fontsize=11)
plt.title('Đánh đổi Bias - Variance theo Độ Sâu Cây Quyết Định', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

[Đã hiển thị đồ thị Bias-Variance. Tại depth >= 8, Train AUC tiếp tục tăng lên >0.78 nhưng Validation AUC bão hòa và giảm, xác nhận hiện tượng Overfitting.]


## 6. Tinh Chỉnh Siêu Tham Số & Khóa Ngưỡng Thẩm Định (Bước 14 – 17)
- Thực hiện GridSearch 80 tổ hợp tham số bằng chiến lược Cross-Validation cửa sổ mở rộng (`TimeSeriesSplit(4)`).
- **Khóa ngưỡng phân tầng (Triage Thresholds) trên tập Validation:**
  - $t_1 = 0.1883$: Phân vị thứ 85 (Top 15% đơn hàng rủi ro cao nhất được xếp vào nhóm **P1 - Khẩn cấp**, cần Trưởng ca can thiệp).
  - $t_2 = 0.0651$: Phân vị thứ 45 (Top 55% được xếp vào nhóm **P2 - Cảnh báo**).
  - Các đơn còn lại thuộc nhóm **P3 - Tiêu chuẩn**.
- Chỉ sau khi hoàn toàn khóa tham số và ngưỡng, mô hình mới được mở tập kiểm thử (Test) **đúng một lần**.

In [ ]:
with open('outputs/04_final_metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

with open('weights/thresholds.json', 'r', encoding='utf-8') as f:
    th = json.load(f)

print("KẾT QUẢ HUẤN LUYỆN CUỐI CÙNG TRÊN TẬP TEST:")
print(f"  - Test ROC-AUC:    {metrics['roc_auc']:.4f}")
print(f"  - Test PR-AUC:     {metrics['pr_auc']:.4f}")
print(f"  - Test Brier:      {metrics['brier']:.4f}")
print(f"  - Ngưỡng khóa t1:  {th['t1']:.4f} (Nhóm P1 - Cực kỳ khẩn cấp)")
print(f"  - Ngưỡng khóa t2:  {th['t2']:.4f} (Nhóm P2 - Cảnh báo rủi ro)")
print(f"  - Ngưỡng F1 tối ưu:{th['t_f1']:.4f}")

KẾT QUẢ HUẤN LUYỆN CUỐI CÙNG TRÊN TẬP TEST:
  - Test ROC-AUC:    0.6553
  - Test PR-AUC:     0.1548
  - Test Brier:      0.0711
  - Ngưỡng khóa t1:  0.1883 (Nhóm P1 - Cực kỳ khẩn cấp)
  - Ngưỡng khóa t2:  0.0651 (Nhóm P2 - Cảnh báo rủi ro)
  - Ngưỡng F1 tối ưu:0.1799


## 7. Kiểm Định Thống Kê & Phân Tích Lỗi (Bước 18 – 20)
- **Bootstrap 1,000 lần:** Khoảng tin cậy 95% của ROC-AUC trên tập test là $[0.6406, 0.6700]$.
- **Bootstrap có cặp (Paired Bootstrap):** Chênh lệch giữa Cây quyết định và Luật người bán $\Delta \text{AUC} = -0.0159$ với khoảng tin cậy $[-0.0242, -0.0076]$. Kết quả chứng minh rõ ràng: Cây không vượt trội hơn luật đơn biến về khả năng phân tách, do đó giá trị của Cây là năng lực phân loại quy tắc trực quan và hỗ trợ quyết định (Triage).

In [ ]:
with open('outputs/05_bootstrap_ci.json', 'r', encoding='utf-8') as f:
    boot = json.load(f)

auc_ci = boot['tree_roc_auc_95_ci']
pr_ci = boot['tree_pr_auc_95_ci']
diff_ci = boot['paired_diff_tree_minus_seller_95_ci']

print("Khoảng tin cậy Bootstrap 95% (Test set, 1,000 lần lặp):")
print(f"  - Decision Tree ROC-AUC: {auc_ci['mean']:.4f}  [95% CI: {auc_ci['ci_lower']:.4f} - {auc_ci['ci_upper']:.4f}]")
print(f"  - Decision Tree PR-AUC:  {pr_ci['mean']:.4f}  [95% CI: {pr_ci['ci_lower']:.4f} - {pr_ci['ci_upper']:.4f}]")
print(f"  - Hiệu (Tree - Seller Rule): {diff_ci['mean']:.4f} [95% CI: {diff_ci['ci_lower']:.4f} - {diff_ci['ci_upper']:.4f}]")

Khoảng tin cậy Bootstrap 95% (Test set, 1,000 lần lặp):
  - Decision Tree ROC-AUC: 0.6548  [95% CI: 0.6395 - 0.6714]
  - Decision Tree PR-AUC:  0.1551  [95% CI: 0.1420 - 0.1699]
  - Hiệu (Tree - Seller Rule): -0.0161 [95% CI: -0.0237 - -0.0082]


## 8. Mô Phỏng Hàng Đợi Vận Hành Kho (As-Is vs To-Be) (Bước 21)
Mô phỏng xử lý 18,765 đơn hàng trong 91 ngày của tập Test tại trung tâm phân loại kho Olist:
- $K=3$ nhân viên xử lý; ca làm việc 08:00 – 17:00 từ Thứ Hai đến Thứ Bảy; thời gian xử lý: $T = 6 + 3 \cdot (\text{n\_items} - 1)$ phút.
- **FIFO (Hiện trạng As-Is):** Tỷ lệ trễ 7.2% (rất sát với tỷ lệ trễ thực tế 7.5% trong dữ liệu).
- **EDD (Luật hạn giao):** Giảm tỷ lệ trễ xuống 0.8%.
- **AI-EDD (Đề xuất To-Be):** Tỷ lệ trễ 0.9%, lead time trung bình 33.2 giờ.
- **AI level-first (Rủi ro cảnh báo):** Nếu lạm dụng AI xếp đơn P1 làm trước mà bỏ qua hạn cam kết, tỷ lệ trễ tăng vọt lên **11.6%**, chứng minh nguy cơ tai hại khi ứng dụng AI sai cách.

In [ ]:
with open('outputs/05_simulation_grid.json', 'r', encoding='utf-8') as f:
    sim_grid = json.load(f)

rows = []
for k_str, tf_dict in sim_grid.items():
    k_val = int(k_str.replace('K=', ''))
    for tf_str, pol_dict in tf_dict.items():
        tf_val = float(tf_str.replace('time_factor=', ''))
        rows.append({
            'K': k_val,
            'Factor': tf_val,
            'FIFO Trễ (%)': round(pol_dict['FIFO']['late_rate'] * 100, 2),
            'EDD Trễ (%)': round(pol_dict['EDD']['late_rate'] * 100, 2),
            'AI-EDD Trễ (%)': round(pol_dict['AI_EDD']['late_rate'] * 100, 2),
            'AI Level-first Trễ (%)': round(pol_dict['AI_level_first']['late_rate'] * 100, 2),
            'FIFO Lead TB (h)': pol_dict['FIFO']['lead_mean_h'],
            'AI-EDD Lead TB (h)': pol_dict['AI_EDD']['lead_mean_h']
        })

sim_df = pd.DataFrame(rows)
k3_df = sim_df[sim_df['K'] == 3]
print("KẾT QUẢ MÔ PHỎNG VẬN HÀNH KHO VỚI K=3 NHÂN VIÊN:")
print(k3_df.to_string(index=False))

KẾT QUẢ MÔ PHỎNG VẬN HÀNH KHO VỚI K=3 NHÂN VIÊN:
 K  Factor  FIFO Trễ (%)  EDD Trễ (%)  AI-EDD Trễ (%)  AI Level-first Trễ (%)  FIFO Lead TB (h)  AI-EDD Lead TB (h)
 3     0.8          0.98         0.77            0.79                    4.00             16.12               16.08
 3     0.9          7.00         0.80            0.80                   11.42             32.57               32.54
 3     1.0          7.24         0.80            0.87                   11.63             33.23               33.19


## 9. Lớp Luật Mô Phỏng Nghiệp Vụ Kho (FEFO & Safety Stock)
Theo quy định Section 10 (Điều cấm), các thuộc tính không có trong Olist (Hạn sử dụng, lô hàng, tồn kho an toàn) được tạo bằng script mô phỏng độc lập `src/synth.py` (`SEED=42`), mang tiền tố `sim_` và lưu tại `data/simulated/`.
**Tuyệt đối không đưa các biến giả lập này vào mô hình huấn luyện ML**, mà chỉ sử dụng làm lớp cảnh báo FEFO và tồn kho cho Trưởng ca trên giao diện vận hành.

In [ ]:
sim_orders = pd.read_csv('data/simulated/orders_simulated_features.csv')
print(f"Tổng số đơn mô phỏng nghiệp vụ kho: {len(sim_orders):,} đơn")
print(f"Số đơn có cảnh báo FEFO (hạn còn <= 3 ngày): {sim_orders['sim_fefo_priority_flag'].sum():,} đơn")
print(sim_orders[['order_id', 'sim_has_perishable_item', 'sim_min_days_to_expiry', 'sim_fefo_priority_flag', 'sim_delay_penalty_cost_brl']].head())

Tổng số đơn mô phỏng nghiệp vụ kho: 98,666 đơn
Số đơn có cảnh báo FEFO (hạn còn <= 3 ngày): 752 đơn
                           order_id  sim_has_perishable_item  sim_min_days_to_expiry  sim_fefo_priority_flag  sim_delay_penalty_cost_brl
0  00010242fe8c5a6d1ba2dd792cb16214                    False                     NaN                   False                        5.89
1  00018f77f2f0320c557190d7a144bdd3                    False                     NaN                   False                       23.99
2  000229ec398224ef6ca0657da4fc703e                    False                     NaN                   False                       19.90
3  00024acbcdf0a6daa1e931b038114c75                    False                     NaN                   False                        1.30
4  00042b26cf59d7ce69dfabb4e55b4fd9                    False                     NaN                   False                       19.99


## 10. Suy Luận & Triển Khai Giao Diện Web (Deployment)
- Script `predict_new.py`: Cho phép nạp bất kỳ file CSV đơn hàng mới nào để đưa ra điểm rủi ro, phân hạng P1/P2/P3 và cờ cảnh báo Trưởng ca.
- Giao diện Web tĩnh (`web/index.html`): Hoàn chỉnh 8 trang tương tác, sử dụng `web/js/tree.js` để suy luận trực tiếp trong trình duyệt bằng pure JavaScript (khớp 100% với Python).

In [ ]:
# Kiểm tra nhanh logic dự đoán mẫu bằng tree.json đã xuất
with open('weights/tree.json', 'r', encoding='utf-8') as f:
    tree_dict = json.load(f)

print(f"Cây quyết định đã nạp thành công: {len(tree_dict['nodes'])} nút")
print(f"Ngưỡng khóa: t1 = {tree_dict['t1']:.4f}, t2 = {tree_dict['t2']:.4f}")
print("Hệ thống sẵn sàng phục vụ triển khai!")

Cây quyết định đã nạp thành công: 81 nút
Ngưỡng khóa: t1 = 0.1883, t2 = 0.0651
Hệ thống sẵn sàng phục vụ triển khai!


## 11. Kết Luận, Hạn Chế & Khuyến Nghị Thực Tiễn
1. **Tính trung thực số liệu:** Mô hình AI không phải là cây gậy thần kỳ có thể loại bỏ hoàn toàn việc trễ hàng. AI có vai trò then chốt như một **bộ lọc cảnh báo sớm (Early-Warning Triage)** giúp Trưởng ca phát hiện 35% đơn trễ tiềm năng trong 15% khối lượng công việc khẩn cấp nhất.
2. **Quy tắc phối hợp vận hành:** Luôn sử dụng quy tắc hạn giao (EDD) làm xương sống điều phối hàng đợi; chỉ áp dụng AI để điều chỉnh độ ưu tiên cục bộ ($\Delta = 24$h), tuyệt đối không để AI lấn át thời hạn cam kết khách hàng.
3. **Giới hạn nghiên cứu:** Dữ liệu Olist phản ánh sàn marketplace phân tán (kho đối tác của seller), không phải mô hình kho tự doanh nguyên khối. Các biến FEFO và tồn kho là mô phỏng giả lập có kiểm soát để phục vụ học thuật môn E-Logistics.

---
*Tài liệu được sinh tự động bởi `build_notebook.py` — Bảo toàn tính nhất quán và khả năng tái lập 100%.*